In [1]:
import os
import pandas as pd
import numpy as np




In [2]:
def processAndWriteDf(df, out_path="./submission.csv"):
    required_cols = ["image_id", "healthy", "multiple_diseases", "rust", "scab"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Submission df missing required columns: {missing}. Found: {list(df.columns)}"
        )

    df_out = df[required_cols].copy()
    df_out.to_csv(out_path, index=False)
    print(f"Wrote submission to: {out_path} (shape={df_out.shape})")
    return df_out




In [3]:
def _softmax(logits: np.ndarray, axis: int = 1) -> np.ndarray:
    z = logits - np.max(logits, axis=axis, keepdims=True)
    ez = np.exp(z)
    return ez / np.sum(ez, axis=axis, keepdims=True)


def _sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-x))


def _resolve_base_paths():
    base = "../input/plant-pathology-2020-fgvc7"
    sample_path = os.path.join(base, "sample_submission.csv")
    test_path = os.path.join(base, "test.csv")
    train_path = os.path.join(base, "train.csv")
    images_dir = os.path.join(base, "images")

    if not os.path.exists(sample_path):
        sample_path = "../input/sample_submission.csv"
    if not os.path.exists(test_path):
        test_path = "../input/test.csv"
    if not os.path.exists(train_path):
        train_path = "../input/train.csv"
    if not os.path.exists(images_dir):
        images_dir = "../input/images"

    return sample_path, test_path, train_path, images_dir


def _standardize_fit(X: np.ndarray, eps: float = 1e-6):
    mu = np.mean(X, axis=0, keepdims=True)
    sd = np.std(X, axis=0, keepdims=True)
    return mu, sd + eps


def _standardize_apply(X: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    return (X - mu) / sd


def _extract_features_from_ids(
    image_ids, images_dir, Image, size=(128, 128)
) -> np.ndarray:
    feats = np.zeros((len(image_ids), 22), dtype=np.float32)

    for i, img_id in enumerate(image_ids):
        img_path = os.path.join(images_dir, f"{img_id}.jpg")
        if not os.path.exists(img_path):
            continue
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize(size)
            arr = np.asarray(img, dtype=np.float32) / 255.0  # (H,W,3)

            mean_rgb = arr.mean(axis=(0, 1))
            std_rgb = arr.std(axis=(0, 1))
            feats[i, 0:3] = mean_rgb
            feats[i, 3:6] = std_rgb

            r = arr[:, :, 0]
            g = arr[:, :, 1]
            b = arr[:, :, 2]

            cmax = np.maximum(np.maximum(r, g), b)
            cmin = np.minimum(np.minimum(r, g), b)
            delta = cmax - cmin

            v = cmax
            s = np.where(cmax > 1e-8, delta / (cmax + 1e-8), 0.0)

            h = np.zeros_like(cmax)
            mask = delta > 1e-8

            mr = mask & (cmax == r)
            mg = mask & (cmax == g)
            mb = mask & (cmax == b)

            h[mr] = ((g[mr] - b[mr]) / (delta[mr] + 1e-8)) % 6.0
            h[mg] = ((b[mg] - r[mg]) / (delta[mg] + 1e-8)) + 2.0
            h[mb] = ((r[mb] - g[mb]) / (delta[mb] + 1e-8)) + 4.0
            h = h / 6.0  # [0,1)

            hsv = np.stack([h, s, v], axis=2)
            mean_hsv = hsv.mean(axis=(0, 1))
            std_hsv = hsv.std(axis=(0, 1))
            feats[i, 6:9] = mean_hsv
            feats[i, 9:12] = std_hsv

            yellow_mask = (r > 0.45) & (g > 0.45) & (b < 0.35)
            yellow_ratio = float(yellow_mask.mean())

            brown_mask = (r > 0.35) & (g > 0.20) & (g < 0.60) & (b < 0.30)
            brown_ratio = float(brown_mask.mean())

            feats[i, 12] = yellow_ratio
            feats[i, 13] = brown_ratio

            exg = float((2.0 * mean_rgb[1]) - mean_rgb[0] - mean_rgb[2])
            feats[i, 14] = exg

            denom = float(mean_rgb[1] + mean_rgb[0] + 1e-6)
            gmr = float((mean_rgb[1] - mean_rgb[0]) / denom)
            feats[i, 15] = gmr

            sat_mask = s > 0.10
            if np.any(sat_mask):
                hh = h[sat_mask].ravel()
                bins = np.minimum((hh * 6.0).astype(np.int32), 5)
                hist = np.bincount(bins, minlength=6).astype(np.float32)
                hist = hist / (hist.sum() + 1e-6)
                feats[i, 16:22] = hist
            else:
                feats[i, 16:22] = 0.0

        except Exception:
            continue

    return feats.astype(np.float32)


def _roc_auc_binary(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y = y_true.astype(np.int32, copy=False)
    s = y_score.astype(np.float64, copy=False)

    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return 0.5

    order = np.argsort(s, kind="mergesort")
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(s) + 1, dtype=np.float64)

    s_sorted = s[order]
    i = 0
    while i < len(s_sorted):
        j = i + 1
        while j < len(s_sorted) and s_sorted[j] == s_sorted[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j

    sum_ranks_pos = ranks[y == 1].sum()
    auc = (sum_ranks_pos - n1 * (n1 + 1) / 2.0) / (n0 * n1)
    return float(auc)


def _pav_weighted_non_decreasing(v: np.ndarray, w: np.ndarray):
    v = np.asarray(v, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)
    n = int(len(v))
    if n == 0:
        return v, w

    lvl = np.empty(n, dtype=np.float64)
    wt = np.empty(n, dtype=np.float64)
    start = np.empty(n, dtype=np.int64)
    end = np.empty(n, dtype=np.int64)

    m = 0
    for i in range(n):
        lvl[m] = v[i]
        wt[m] = w[i]
        start[m] = i
        end[m] = i
        m += 1

        while m >= 2 and lvl[m - 2] > lvl[m - 1] + 1e-15:
            new_w = wt[m - 2] + wt[m - 1]
            new_lvl = (lvl[m - 2] * wt[m - 2] + lvl[m - 1] * wt[m - 1]) / (
                new_w + 1e-18
            )
            wt[m - 2] = new_w
            lvl[m - 2] = new_lvl
            end[m - 2] = end[m - 1]
            m -= 1

    out_v = np.empty(n, dtype=np.float64)
    out_w = np.empty(n, dtype=np.float64)
    for j in range(m):
        out_v[start[j] : end[j] + 1] = lvl[j]
        out_w[start[j] : end[j] + 1] = wt[j] / max(1, (end[j] - start[j] + 1))
    return out_v, out_w


def _fit_label_aware_monotone_mapper(
    train_scores_1d: np.ndarray,
    train_labels_1d: np.ndarray,
    n_bins: int = 200,
    prior_strength: float = 1.0,
):
    s = train_scores_1d.astype(np.float64, copy=False)
    y = train_labels_1d.astype(np.float64, copy=False)

    n = len(s)
    if n == 0:
        knots = np.array([-1.0, 1.0], dtype=np.float64)
        probs = np.array([0.5, 0.5], dtype=np.float64)
        return knots, probs

    order = np.argsort(s, kind="mergesort")
    s_sorted = s[order]
    y_sorted = y[order]

    B = int(max(10, min(n_bins, n)))
    edges = np.linspace(0, n, B + 1, dtype=np.int64)

    bin_score = np.zeros(B, dtype=np.float64)
    bin_p = np.zeros(B, dtype=np.float64)
    w = np.zeros(B, dtype=np.float64)

    base_rate = float(np.clip(y.mean(), 1e-6, 1.0 - 1e-6))
    a0 = prior_strength * base_rate
    b0 = prior_strength * (1.0 - base_rate)

    for b in range(B):
        lo = int(edges[b])
        hi = int(edges[b + 1])
        if hi <= lo:
            hi = min(n, lo + 1)

        ss = s_sorted[lo:hi]
        yy = y_sorted[lo:hi]
        bin_score[b] = float(np.median(ss))
        pos = float(yy.sum())
        cnt = float(len(yy))
        w[b] = cnt
        bin_p[b] = (pos + a0) / (cnt + a0 + b0)

    bin_p_mono, w_mono = _pav_weighted_non_decreasing(bin_p, w)

    s_knots = np.concatenate(([bin_score[0] - 1.0], bin_score, [bin_score[-1] + 1.0]))
    p_knots = np.concatenate(([bin_p_mono[0]], bin_p_mono, [bin_p_mono[-1]]))
    return s_knots, p_knots


def _apply_monotone_mapper(
    test_scores_1d: np.ndarray, s_knots: np.ndarray, p_knots: np.ndarray
):
    s = test_scores_1d.astype(np.float64, copy=False)
    p = np.interp(s, s_knots, p_knots)
    return np.clip(p, 1e-6, 1.0 - 1e-6)


def _make_stratified_split_multilabel(
    Y: np.ndarray, val_frac: float = 0.2, seed: int = 1337, min_val: int = 200
):
    # Minimal change to improve stability of AUC-based selection:
    # deterministic greedy assignment to roughly preserve per-class prevalence in val.
    n = Y.shape[0]
    n_val = max(min_val, int(val_frac * n))
    rng = np.random.RandomState(seed)

    # Sort by label density so multi-positive rows get placed first.
    density = Y.sum(axis=1)
    order = np.argsort(-density + 1e-3 * rng.randn(n))

    target_pos = Y.sum(axis=0) * (n_val / n)
    val_pos = np.zeros(Y.shape[1], dtype=np.float64)
    val_idx = []
    tr_idx = []

    for i in order:
        if len(val_idx) >= n_val:
            tr_idx.append(i)
            continue
        y = Y[i]
        # Score how much this row helps reach target positives; keep deterministic.
        gain = np.sum(
            np.minimum(target_pos, val_pos + y) - np.minimum(target_pos, val_pos)
        )
        # Add small penalty if it overshoots too much.
        overshoot = np.sum(np.maximum(0.0, (val_pos + y) - target_pos))
        score = gain - 0.1 * overshoot
        if score >= 0 or (n_val - len(val_idx)) > (
            len(order) - np.where(order == i)[0][0] - 1
        ):
            val_idx.append(i)
            val_pos += y
        else:
            tr_idx.append(i)

    # If still short due to heuristic, fill randomly from remaining.
    if len(val_idx) < n_val:
        remaining = np.setdiff1d(
            np.arange(n), np.array(val_idx, dtype=np.int64), assume_unique=False
        )
        rng.shuffle(remaining)
        need = n_val - len(val_idx)
        val_idx.extend(list(remaining[:need]))
        tr_idx = list(
            np.setdiff1d(
                np.arange(n), np.array(val_idx, dtype=np.int64), assume_unique=False
            )
        )

    return np.array(tr_idx, dtype=np.int64), np.array(val_idx, dtype=np.int64)




In [4]:
def getPredictionFromTPUModel():
    sample_path, test_path, train_path, images_dir = _resolve_base_paths()

    sample_sub = pd.read_csv(sample_path)
    test_df = pd.read_csv(test_path)
    train_df = pd.read_csv(train_path)

    target_cols = ["healthy", "multiple_diseases", "rust", "scab"]
    for c in ["image_id"] + target_cols:
        if c not in sample_sub.columns:
            raise ValueError(
                f"sample_submission.csv missing column '{c}'. Found: {list(sample_sub.columns)}"
            )

    try:
        from PIL import Image
    except Exception as e:
        print(f"PIL import failed ({e}); falling back to uniform predictions.")
        preds = np.full((len(test_df), 4), 0.25, dtype=np.float32)
        pred_df = pd.DataFrame(preds, columns=target_cols)
        result = pd.concat([test_df[["image_id"]].copy(), pred_df], axis=1)
        result = result[sample_sub.columns]
        return result

    X_train_raw = _extract_features_from_ids(
        train_df["image_id"].values, images_dir, Image
    )
    X_test_raw = _extract_features_from_ids(
        test_df["image_id"].values, images_dir, Image
    )

    def _extend(X):
        Rm, Gm, Bm, Rs, Gs, Bs = [X[:, j] for j in range(6)]
        brightness = (Rm + Gm + Bm) / 3.0
        variance = (Rs + Gs + Bs) / 3.0
        rg = Rm - Gm
        gb = Gm - Bm
        rb = Rm - Bm
        gr = Gm - Rm
        return np.column_stack([X, brightness, variance, rg, gb, rb, gr]).astype(
            np.float32
        )

    X_train_ext_raw = _extend(X_train_raw)
    X_test_ext_raw = _extend(X_test_raw)

    train_row_sum = np.sum(np.abs(X_train_ext_raw), axis=1)
    test_row_sum = np.sum(np.abs(X_test_ext_raw), axis=1)
    train_mean_feat = (
        X_train_ext_raw[train_row_sum > 0].mean(axis=0, keepdims=True)
        if np.any(train_row_sum > 0)
        else np.zeros((1, X_train_ext_raw.shape[1]), dtype=np.float32)
    )
    X_train_ext_raw = X_train_ext_raw.copy()
    X_test_ext_raw = X_test_ext_raw.copy()
    X_train_ext_raw[train_row_sum == 0] = train_mean_feat
    X_test_ext_raw[test_row_sum == 0] = train_mean_feat

    mu, sd = _standardize_fit(X_train_ext_raw)
    X_train_ext = _standardize_apply(X_train_ext_raw, mu, sd).astype(np.float32)
    X_test_ext = _standardize_apply(X_test_ext_raw, mu, sd).astype(np.float32)

    Y_train = train_df[target_cols].values.astype(np.float32)

    Xtr_full = np.concatenate(
        [np.ones((X_train_ext.shape[0], 1), dtype=np.float32), X_train_ext], axis=1
    )
    Xte = np.concatenate(
        [np.ones((X_test_ext.shape[0], 1), dtype=np.float32), X_test_ext], axis=1
    )

    # Change (score-relevant): use a deterministic stratified split for more reliable AUC-based hyperparameter selection.
    tr_idx, val_idx = _make_stratified_split_multilabel(
        Y_train, val_frac=0.2, seed=1337, min_val=200
    )

    Xtr = Xtr_full[tr_idx].astype(np.float64, copy=False)
    Ytr = Y_train[tr_idx].astype(np.float64, copy=False)
    Xva = Xtr_full[val_idx].astype(np.float64, copy=False)
    Yva = Y_train[val_idx].astype(np.float64, copy=False)

    Xt = Xtr.T
    XtX = Xt @ Xtr
    I = np.eye(XtX.shape[0], dtype=np.float64)
    I[0, 0] = 0.0  # do not regularize bias

    lam_grid = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
    best_lam = lam_grid[0]
    best_auc = -1.0

    for lam in lam_grid:
        A = XtX + float(lam) * I
        W = np.zeros((Xtr.shape[1], Ytr.shape[1]), dtype=np.float64)
        for k in range(Ytr.shape[1]):
            W[:, k] = np.linalg.solve(A, Xt @ Ytr[:, k])

        va_scores = Xva @ W  # raw linear scores
        aucs = []
        for k in range(Ytr.shape[1]):
            aucs.append(_roc_auc_binary(Yva[:, k], va_scores[:, k]))
        mean_auc = float(np.mean(aucs))
        if mean_auc > best_auc:
            best_auc = mean_auc
            best_lam = lam

    # Fit final ridge on ALL training data using best_lam (same core logic, closed-form).
    Xtr64 = Xtr_full.astype(np.float64, copy=False)
    Y64 = Y_train.astype(np.float64, copy=False)
    Xt_full = Xtr64.T
    XtX_full = Xt_full @ Xtr64
    I_full = np.eye(XtX_full.shape[0], dtype=np.float64)
    I_full[0, 0] = 0.0
    A_full = XtX_full + float(best_lam) * I_full

    W_full = np.zeros((Xtr64.shape[1], Y64.shape[1]), dtype=np.float64)
    for k in range(Y64.shape[1]):
        W_full[:, k] = np.linalg.solve(A_full, Xt_full @ Y64[:, k])

    train_scores_full = (Xtr64 @ W_full).astype(np.float64)
    test_scores = (Xte.astype(np.float64, copy=False) @ W_full).astype(np.float64)

    # Select calibrator hyperparams on the same split, but (score-relevant) refit final calibrator on ALL training scores.
    cal_bins_grid = [80, 120, 200]
    prior_grid = [0.5, 1.0, 2.0]
    best_cal = (200, 1.0)
    best_cal_auc = -1.0

    tr_scores = train_scores_full[tr_idx]
    va_scores = train_scores_full[val_idx]
    Ytr_cal = Y_train[tr_idx].astype(np.float64, copy=False)
    Yva_cal = Y_train[val_idx].astype(np.float64, copy=False)

    for nb in cal_bins_grid:
        for ps in prior_grid:
            aucs = []
            for k in range(Y_train.shape[1]):
                s_knots, p_knots = _fit_label_aware_monotone_mapper(
                    tr_scores[:, k],
                    Ytr_cal[:, k],
                    n_bins=int(nb),
                    prior_strength=float(ps),
                )
                va_prob = _apply_monotone_mapper(va_scores[:, k], s_knots, p_knots)
                aucs.append(_roc_auc_binary(Yva_cal[:, k], va_prob))
            mean_auc = float(np.mean(aucs))
            if mean_auc > best_cal_auc:
                best_cal_auc = mean_auc
                best_cal = (int(nb), float(ps))

    # Change (score-relevant): refit monotone mapper on ALL training scores from the final ridge model for distribution consistency.
    probs = np.zeros_like(test_scores, dtype=np.float64)
    for k in range(Y_train.shape[1]):
        s_knots, p_knots = _fit_label_aware_monotone_mapper(
            train_scores_full[:, k],
            Y_train[:, k].astype(np.float64, copy=False),
            n_bins=best_cal[0],
            prior_strength=best_cal[1],
        )
        probs[:, k] = _apply_monotone_mapper(test_scores[:, k], s_knots, p_knots)

    probs = probs.astype(np.float32)
    probs = np.clip(probs, 1e-6, 1.0 - 1e-6)

    pred_df = pd.DataFrame(probs, columns=target_cols)
    result = pd.concat([test_df[["image_id"]].copy(), pred_df], axis=1)
    result = result[sample_sub.columns]
    return result




In [5]:
isTPU = True

if isTPU:
    df = getPredictionFromTPUModel()
    print(df.head(5))
    processAndWriteDf(df, out_path="./submission.csv")
else:
    df = pd.read_csv("../input/notebook45bc751087/submission.csv")
    processAndWriteDf(df, out_path="./submission.csv")

  image_id   healthy  multiple_diseases      rust      scab
0   Test_0  0.243405           0.225249  0.520092  0.150562
1   Test_1  0.337889           0.024293  0.320973  0.214094
2   Test_2  0.337889           0.024293  0.393710  0.214094
3   Test_3  0.083393           0.001240  0.393710  0.650631
4   Test_4  0.353189           0.001240  0.635475  0.056732
Wrote submission to: ./submission.csv (shape=(183, 5))
